# Stage 3 — Four-way RAG Evaluation

Evaluate identical questions under four conditions: `standard`, `llm_pipeline`, `bert`, and `oracle`.

The final RAG evaluation reports **three metrics only**:

1. **Expected Behavior** (`expected_behavior_adherence`): whether an answer follows the gold conflict type's expected response behavior.
2. **Answer Recall** (`answer_recall`): whether the answer includes the official reference answer, when one exists.
3. **Factual Grounding** (`factual_grounding`): the fraction of factual claims supported by retrieved documents.

Behavior, Grounding and Recall are evaluated independently, with fingerprinted resumable caches. Python segments and numbers the Grounding claims; GPT-4o evaluates each fixed `claim_id` using strict structured output. Identical answers reuse Judge evaluations.

The framework is adapted from the [CONFLICTS study](https://arxiv.org/html/2506.08500); this is an independent implementation rather than a reproduction of the paper's numeric results. Freeze the evaluation protocol before Test; report the absence of human Judge validation as a limitation when applicable.


## 1. Setup and output paths

Place the Stage 2 JSONL, frozen dataset, and split manifest in the workspace. For Answer Recall, also provide the unmodified official `conflicts.jsonl` (or `conflicts_458.jsonl`).

Previously saved Judge caches remain reusable: older Grounding entries store the factual metric as `factual_support`, which is mapped to `factual_grounding` on export. Citation fields from old caches are excluded from all new scoring outputs.


In [1]:
import os, json, hashlib, time, getpass, re
from pathlib import Path
import numpy as np
import pandas as pd
from openai import OpenAI
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
EVAL_MODE = "final_test"  # Run Test evaluation only.
VALIDATION_SMOKE_PER_CLASS = 1    # Must match Stage 2 if using smoke.
RUN_JUDGE_API = True           # Set True only for missing Judge evaluations.
EVALUATOR_MODEL = "gpt-4o-2024-08-06"
RUBRIC_VERSION = "expected_behavior_blind"
GROUNDING_VERSION = "claim_ids_strict_schema"
GROUNDING_SPLITTER_VERSION = "line_and_sentence_boundaries"
GROUNDING_CLAIMS_PER_CALL = 12  # Maximum number of claims in each strict-schema request.
RECALL_VERSION = "official_reference_only"
FROZEN_EVALUATION_PROTOCOL = True  # Must explicitly confirm for final_test.

WORKSPACE = Path.cwd()
IS_KAGGLE = Path("/kaggle/input").is_dir()
OUTPUT_DIR = Path("/kaggle/working") if IS_KAGGLE else WORKSPACE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Use data files in the current working directory only
DATA_DIR = Path.cwd()

if not (DATA_DIR / "conflicts_621.jsonl").is_file():
    raise FileNotFoundError("Missing conflicts_621.jsonl")

if not (DATA_DIR / "conflicts_original_eval_split.json").is_file():
    raise FileNotFoundError("Missing split manifest")

DATA_PATH = DATA_DIR / "conflicts_621.jsonl"
SPLIT_PATH = DATA_DIR / "conflicts_original_eval_split.json"
# An unmodified original data file is needed for reference-based Recall.
OFFICIAL_ORIGINAL_PATH = DATA_DIR / "conflicts.jsonl"
if not OFFICIAL_ORIGINAL_PATH.is_file() and (DATA_DIR / "conflicts_458.jsonl").is_file():
    OFFICIAL_ORIGINAL_PATH = DATA_DIR / "conflicts_458.jsonl"
if not OFFICIAL_ORIGINAL_PATH.is_file():
    raise FileNotFoundError(
        "Original conflicts_458.jsonl (or conflicts.jsonl) is required for "
        "the date-aware Test evidence and reference answers"
    )
ORIGINAL_DATA_SHA256 = hashlib.sha256(OFFICIAL_ORIGINAL_PATH.read_bytes()).hexdigest()
REQUIRE_RECALL_REFERENCE_FOR_FINAL = True  # fail rather than silently report all N/A

GENERATIONS_PATH_OVERRIDE = ""  # Use absolute path if needed.
OUTPUT_DIR = WORKSPACE

GENERATIONS_BASENAMES = {
    "validation_smoke": "rag_validation_smoke_fourway.jsonl",
    "validation_full": "rag_validation_fourway.jsonl",
    "final_test": "rag_test_fourway.jsonl",
}
if EVAL_MODE not in GENERATIONS_BASENAMES:
    raise ValueError("Unknown EVAL_MODE")
if EVAL_MODE == "final_test" and not FROZEN_EVALUATION_PROTOCOL:
    raise RuntimeError("Freeze the evaluation rubric before final Test")
GENERATIONS_PATH = (Path(GENERATIONS_PATH_OVERRIDE).expanduser() if GENERATIONS_PATH_OVERRIDE
                    else OUTPUT_DIR / GENERATIONS_BASENAMES[EVAL_MODE])
RUN_TAG = {"validation_smoke": "smoke", "validation_full": "validation", "final_test": "test"}[EVAL_MODE]
PREFIX = "rag_" + RUN_TAG + "_"
BEHAVIOR_CACHE = OUTPUT_DIR / (PREFIX + "behavior_judgments.jsonl")
GROUNDING_CACHE = OUTPUT_DIR / (PREFIX + "grounding_judgments.jsonl")
RECALL_CACHE = OUTPUT_DIR / (PREFIX + "recall_judgments.jsonl")
SCORES_PATH = OUTPUT_DIR / (PREFIX + "answer_scores.csv")
SUMMARY_PATH = OUTPUT_DIR / (PREFIX + "condition_summary.csv")
PER_CLASS_PATH = OUTPUT_DIR / (PREFIX + "per_class_summary.csv")
PAIRED_PATH = OUTPUT_DIR / (PREFIX + "paired_differences.csv")
CLASSIFIER_PATH = OUTPUT_DIR / (PREFIX + "classifier_comparison.csv")
CLASSIFIER_PREDICTIONS_PATH = OUTPUT_DIR / (PREFIX + "classifier_predictions.csv")
print("Mode:", EVAL_MODE, "| Judge API:", RUN_JUDGE_API)
print("Input:", GENERATIONS_PATH)


Mode: final_test | Judge API: True
Input: /Users/GXM/03-trimester 2/AIML339/Assignment/7/rag-spread 新数据/rag_test_fourway.jsonl


## 2. Load and validate Stage 2 results

Confirm frozen input identity, all four conditions per question, identical generator settings, valid label provenance and identical answers whenever the complete generation request is identical.


In [2]:
def read_jsonl(path):
    if not Path(path).is_file():
        raise FileNotFoundError(f"Missing file: {path}")
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

data = read_jsonl(DATA_PATH)
assert len(data) == 621, "Expected frozen 621-row dataset"
with open(SPLIT_PATH, encoding="utf-8") as f:
    split = json.load(f)
canonical = "\n".join(json.dumps(x, ensure_ascii=False, sort_keys=True) for x in data)
sha = hashlib.sha256(canonical.encode()).hexdigest()
assert sha == split["dataset_sha256"], "Frozen dataset digest mismatch"
assert (len(split["train_indices"]), len(split["validation_indices"]), len(split["test_indices"])) == (481, 68, 72)
assert set(split["train_indices"] + split["validation_indices"] + split["test_indices"]) == set(range(621))
assert all(i < 458 for i in split["validation_indices"] + split["test_indices"])

# Restore the SAME real source-date metadata used by Stage 2 for every Judge.
# Do not change the frozen classification dataset on disk or the split hash.
_original_evidence = read_jsonl(OFFICIAL_ORIGINAL_PATH)
assert len(_original_evidence) == 458

def _usable_source_date(raw):
    if raw is None:
        return None
    value = str(raw).strip()
    return None if value.lower() in {"", "na", "n/a", "nan", "none", "null"} else value

for i in split["validation_indices"] + split["test_indices"]:
    src = _original_evidence[int(i)]
    target = data[int(i)]
    if src["question"] != target["question"] or src["conflict_type"] != target["conflict_type"]:
        raise ValueError(f"Original data misaligned at example_id={i}")
    if len(src["search_results"]) != len(target["search_results"]):
        raise ValueError(f"Original documents misaligned at example_id={i}")
    for j, (source_doc, prepared_doc) in enumerate(zip(src["search_results"], target["search_results"])):
        for field in ("title", "short_text"):
            if str(source_doc.get(field) or "").strip() != str(prepared_doc.get(field) or "").strip():
                raise ValueError(f"Evidence mismatch at example_id={i}, doc={j + 1}, field={field}")
        date = _usable_source_date(source_doc.get("date"))
        if date:
            prepared_doc["date"] = date
print("Date-aware Test evidence restored and aligned with Stage 2")

if EVAL_MODE == "validation_smoke":
    seen = {}
    expected_ids = []
    for i in split["validation_indices"]:
        typ = data[i]["conflict_type"]
        if seen.get(typ, 0) < VALIDATION_SMOKE_PER_CLASS:
            expected_ids.append(i)
            seen[typ] = seen.get(typ, 0) + 1
elif EVAL_MODE == "validation_full":
    expected_ids = list(split["validation_indices"])
else:
    expected_ids = list(split["test_indices"])

CONDITIONS = ("standard", "llm_pipeline", "bert", "oracle")
outputs = pd.DataFrame(read_jsonl(GENERATIONS_PATH))
need = {"example_id", "condition", "answer", "question", "gold_label", "predicted_label",
        "llm_predicted_label", "routed_label", "run_mode", "config_hash", "prompt_version", "router_version",
        "evaluation_protocol_version", "generator_model", "temperature", "max_output_tokens",
        "bert_checkpoint_sha256", "llm_classifier_model", "llm_classifier_config_hash",
        "selected_seed", "seed_selection_sha256", "selection_is_provisional",
        "generation_request_hash", "generation_cache_hit", "prompt", "original_data_sha256"}
assert need.issubset(outputs.columns), f"Missing Stage 2 columns: {need - set(outputs.columns)}"
expected_pairs = {(int(i), c) for i in expected_ids for c in CONDITIONS}
actual_pairs = [(int(x.example_id), x.condition) for x in outputs.itertuples(index=False)]
assert len(actual_pairs) == len(expected_pairs) and set(actual_pairs) == expected_pairs, (
    f"Wrong condition/ID coverage: expected {len(expected_pairs)}, got {len(outputs)}"
)
assert outputs.answer.fillna("").astype(str).str.strip().ne("").all()
for name in ("config_hash", "prompt_version", "router_version", "evaluation_protocol_version", "run_mode",
             "generator_model", "temperature", "max_output_tokens", "bert_checkpoint_sha256",
             "llm_classifier_model", "llm_classifier_config_hash", "selected_seed",
             "seed_selection_sha256", "selection_is_provisional", "original_data_sha256"):
    assert outputs[name].nunique(dropna=False) == 1, f"Mixed Stage 2 setting: {name}"
assert outputs.evaluation_protocol_version.eq("fourway_conflict_aware_rag").all()
assert outputs.prompt_version.eq("rag_with_source_dates_and_sentence_citations").all()
assert outputs.original_data_sha256.eq(ORIGINAL_DATA_SHA256).all(), "Original date evidence file differs from Stage 2"
assert outputs.router_version.eq("taxonomy_type_conditioned").all()
assert outputs.run_mode.eq(EVAL_MODE).all()
assert outputs.max_output_tokens.eq(500).all()
if EVAL_MODE == "final_test":
    assert outputs.selection_is_provisional.eq(False).all(), "Final BERT cannot be provisional"
    assert outputs.selected_seed.notna().all(), "Final must record selected seed"
    assert outputs.seed_selection_sha256.notna().all(), "Missing Validation seed-selection evidence"
for row in outputs.itertuples(index=False):
    source = data[int(row.example_id)]
    assert row.question == source["question"] and row.gold_label == source["conflict_type"]
for example_id, sub in outputs.groupby("example_id"):
    assert len(sub) == 4 and set(sub.condition) == set(CONDITIONS)
    assert sub.gold_label.nunique() == sub.predicted_label.nunique() == sub.llm_predicted_label.nunique() == 1
    gold = sub.gold_label.iloc[0]; bert = sub.predicted_label.iloc[0]; llm = sub.llm_predicted_label.iloc[0]
    assert pd.isna(sub.loc[sub.condition == "standard", "routed_label"].iloc[0])
    assert sub.loc[sub.condition == "bert", "routed_label"].iloc[0] == bert
    assert sub.loc[sub.condition == "llm_pipeline", "routed_label"].iloc[0] == llm
    assert sub.loc[sub.condition == "oracle", "routed_label"].iloc[0] == gold
# Verify identical complete generation requests always reuse the exact same answer.
for request_hash, group in outputs.groupby("generation_request_hash"):
    assert group["answer"].nunique() == 1, f"Same request produced different answers: {request_hash}"
    assert group["prompt"].nunique() == 1, f"Hash collision or corrupt prompt: {request_hash}"
    expected_request_hash = hashlib.sha256(json.dumps({
        "prompt": group["prompt"].iloc[0],
        "model": group["generator_model"].iloc[0],
        "temperature": int(group["temperature"].iloc[0]),
        "max_output_tokens": int(group["max_output_tokens"].iloc[0]),
    }, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
    assert request_hash == expected_request_hash, "Generator request fingerprint mismatch"
    assert int((group["generation_cache_hit"] == False).sum()) <= 1, "Duplicate billed requests"
print("Verified:", len(outputs), "answers for", len(expected_ids), "unique questions")
display(outputs.condition.value_counts())


Date-aware Test evidence restored and aligned with Stage 2
Verified: 288 answers for 72 unique questions


condition
standard        72
llm_pipeline    72
bert            72
oracle          72
Name: count, dtype: int64

## 3. Restore verified reference answers

Recall uses only official reference answers from the original CONFLICTS file, when present. Never infer missing answers from generated text, and never modify the frozen dataset.


In [3]:
LABELS = [
    "No conflict", "Complementary information", "Conflicting opinions and research outcomes",
    "Conflict due to outdated information", "Conflict due to misinformation",
]
RECALL_LABELS = {"No conflict", "Conflict due to outdated information", "Conflict due to misinformation"}

def valid_reference(value):
    if value is None:
        return None
    if isinstance(value, (list, dict)):
        return None
    text = str(value).strip()
    return text if text and text.lower() not in {"none", "null", "nan", "n/a"} else None

reference_answers = {}
for i in expected_ids:
    value = valid_reference(data[i].get("correct_answer"))
    if value and data[i]["conflict_type"] in RECALL_LABELS:
        reference_answers[int(i)] = value

if OFFICIAL_ORIGINAL_PATH.is_file():
    official = read_jsonl(OFFICIAL_ORIGINAL_PATH)
    assert len(official) == 458, "Official reference file must contain 458 original CONFLICTS rows"
    for i in expected_ids:
        orig, item = official[int(i)], data[int(i)]
        assert orig["question"] == item["question"], f"Original reference question mismatch at ID {i}; stop"
        assert orig["conflict_type"] == item["conflict_type"], f"Original reference label mismatch at ID {i}; stop"
        old_titles = [str(d.get("title") or "").strip().lower() for d in orig.get("search_results", [])]
        new_titles = [str(d.get("title") or "").strip().lower() for d in item.get("search_results", [])]
        if old_titles and new_titles and old_titles != new_titles:
            raise ValueError(f"Original document identity/order mismatch at ID {i}; do NOT match by index")
        if item["conflict_type"] not in RECALL_LABELS:
            continue
        official_answer = valid_reference(orig.get("correct_answer"))
        existing = reference_answers.get(int(i))
        if existing is not None and official_answer is not None:
            assert existing == official_answer, f"Conflicting gold answers for example {i}"
        if official_answer is not None:
            reference_answers[int(i)] = official_answer
    print("Verified official 458-row reference file; reference answers are read-only")
else:
    print("Official original file not found; Recall will be N/A unless frozen data contain correct_answer")

eligible_ids = [i for i in expected_ids if data[int(i)]["conflict_type"] in RECALL_LABELS]
missing_refs = sorted(set(eligible_ids) - set(reference_answers))
print(f"Reference coverage: {len(reference_answers)}/{len(eligible_ids)} eligible questions")
if missing_refs:
    print("Missing reference ID examples:", missing_refs[:15])
if EVAL_MODE == "final_test" and REQUIRE_RECALL_REFERENCE_FOR_FINAL and not reference_answers:
    raise RuntimeError(
        "No authenticated original correct_answer references found. Place original "
        "conflicts.jsonl / conflicts_458.jsonl, or explicitly disable the Recall metric "
        "after deciding in advance and documenting the reason.")


Verified official 458-row reference file; reference answers are read-only
Reference coverage: 35/36 eligible questions
Missing reference ID examples: [400]


## 4. Behavior and evidence rubrics

Behavior follows the gold conflict type. Grounding evaluates whether each factual claim is supported by the retrieved documents, **not** whether its wording matches verbatim. Recall uses verified reference answers when available. Citation formatting is not scored.


In [4]:
TAXONOMY = {
 "No conflict": (
   "Relevant sources provide equivalent or nearly equivalent answers; irrelevant passages are not conflicting answers.",
   "Provide a clear direct supported response; do not fabricate a dispute or unnecessary hedging."),
 "Complementary information": (
   "Different sources give compatible answers or meaningful facets of an underspecified query.",
   "Reconcile the distinct, compatible and QUESTION-RELEVANT answers or interpretations; do not invent an artificial disagreement or require unrelated trivia."),
 "Conflicting opinions and research outcomes": (
   "Relevant retrieved sources disagree in arguments, subjective interpretations or research findings.",
   "Acknowledge the substantive disagreement and fairly communicate the evidence-backed perspectives, without presenting unjustified consensus."),
 "Conflict due to outdated information": (
   "Apparently contradictory facts reflect a change over time; older reports have been superseded.",
   "Prioritize the latest applicable fact supported by retrieved material; do not treat superseded values as equally current. Respect the query's explicit time frame."),
 "Conflict due to misinformation": (
   "One or more documents contain misleading or inaccurate factual answers, not merely different valid scopes or dates.",
   "Prefer the reliable supported fact, avoid repeating disproven alternatives as equally correct, and express uncertainty if evidence genuinely cannot resolve the claim."),
}
assert set(TAXONOMY) == set(LABELS)
# Synthetic examples on new topics, NOT reused from evaluation queries or paper demonstrations.
BEHAVIOR_EXAMPLES = {
 "No conflict": [
  ("Q: When does the park close? Both notices state 6 PM. Good: It closes at 6 PM.",
   "Bad: Because one notice says 18:00 and another 6 PM, the closing time is uncertain."),
  ("Q: Who designed the gallery? Both biographies name the same architect. Good: Name the architect.",
   "Bad: Claim conflicting designers without evidence.")],
 "Complementary information": [
  ("Q: What does an artist membership include? One notice lists workshops; another lists studio access. Good: Mention both relevant benefits together.",
   "Bad: Mention workshops alone despite the explicit question about total membership benefits."),
  ("Q: How can guests reach the island? Ferry and licensed shuttle are both valid routes. Good: Present both options concisely.",
   "Bad: Treat the options as mutually exclusive or give unrelated transport trivia.")],
 "Conflicting opinions and research outcomes": [
  ("Q: Should a city adopt a congestion charge? One report finds less traffic; another predicts harm to small traders. Good: Represent both supported views fairly.",
   "Bad: State that every study proves the same position."),
  ("Q: Are shorter school weeks beneficial? Two credible studies draw different conclusions. Good: State the disagreement and its limits.",
   "Bad: Select one conclusion and say there is no contrary evidence.")],
 "Conflict due to outdated information": [
  ("Q: What is the current library closing time? An old schedule lists 5 PM; updated notice lists 7 PM. Good: Say 7 PM and date the old schedule if useful.",
   "Bad: Present 5 PM and 7 PM as equally current guesses."),
  ("Q: Who manages the hall now? 2021 profile names Alex; 2024 appointment names Sam. Good: Say Sam with latest evidence.",
   "Bad: Say either Alex or Sam might be manager, ignoring recency.")],
 "Conflict due to misinformation": [
  ("Q: Where is a public exhibition held? Official organizer and tickets say Hall A; an unrelated repost falsely says Hall C. Good: Give Hall A using credible evidence.",
   "Bad: Say Hall A and Hall C are equally valid venues."),
  ("Q: Which year did a route open? Official dated records agree 1998; a copied blog misprints 1988. Good: Answer 1998 supported by records.",
   "Bad: Reconcile 1988 and 1998 as two equally true opening years.")],
}
assert set(BEHAVIOR_EXAMPLES) == set(LABELS)

BEHAVIOR_SYSTEM = """You are a blind research evaluator. Your ONLY task is binary Expected Behavior Adherence.
The expert conflict type and corresponding behavior are provided only for EVALUATION.
Assess the candidate answer to the user's specific question in light of the retrieved evidence.
A direct factually supported answer can pass for No conflict, Outdated or Misinformation without listing all sources.
For Complementary: integrate meaningful compatible perspectives needed to address the question's scope, NOT every irrelevant or tangential retrieved fact. Do not punish a concise answer for excluding unrelated trivia.
For Opinions: judge whether the material evidence-backed disagreement is fairly presented.
Do not reward sheer length. Answer correctness alone does not automatically establish behavior adherence.
Illustrative good/bad examples are behavioral aids, NEVER evidence about the real question.
Return strict JSON: {"adherence": 0, "reason": "short explanation", "missing_behavior": "brief or empty"}.
Use adherence 1 only if the answer follows the specified type-specific behavior; otherwise 0.
"""

def format_documents(documents):
    parts = []
    for i, doc in enumerate(documents, start=1):
        lines = [f"[Document {i}]"]
        for title, key in [("Title", "title"), ("Source date", "date"),
                           ("Relevant passage", "short_text")]:
            value = str(doc.get(key) or "").strip()
            if value:
                lines.append(f"{title}: {value}")
        parts.append("\n".join(lines))
    return "\n\n".join(parts)

def behavior_input(row):
    item = data[int(row["example_id"])]
    label = item["conflict_type"]
    definition, expected = TAXONOMY[label]
    examples = "\n".join(f"Positive: {pos}\nNegative: {neg}" for pos,neg in BEHAVIOR_EXAMPLES[label])
    return (f"QUESTION:\n{item['question']}\n\nGOLD TYPE (evaluation only): {label}\n"
            f"DEFINITION: {definition}\nEXPECTED BEHAVIOR: {expected}\n\n"
            f"BEHAVIORAL EXAMPLES (not evidence):\n{examples}\n\n"
            f"RETRIEVED DOCUMENTS:\n{format_documents(item['search_results'])}\n\n"
            f"CANDIDATE ANSWER:\n{row['answer']}")

GROUNDING_SYSTEM = """You are a blind evidence-based factual claim assessor, independent of the conflict-type behavior score.
Python has already segmented the candidate answer into claims and supplied fixed integer claim_ids.
Assess ONLY the listed claim_ids exactly once each. Never split, merge, invent, or skip claim_ids.
The output structure is fixed by JSON Schema: fill in each REQUIRED claim_{id} object.
Use only retrieved documents; no external knowledge. Assess whether ALL material factual content in each claim is supported.
'supported': all material factual content is supported by one or more retrieved documents, whether or not cited inline.
'unsupported': documents provide insufficient evidence.
'contradicted': retrieved evidence directly contradicts the claim.
'nonfactual': purely rhetorical or structural text with no factual claim.
evidence_doc_ids: 1-based retrieved document indices that actually SUPPORT the claim (nonempty ONLY for supported; [] otherwise).
Inline citations like [2] are NOT by themselves proof; verify actual evidence. Do not add sentence text or free-form explanations.
"""

# Deterministic splitting: Python owns each assessed span and its claim ID.
# Preserve punctuation and inline [n] citations with the original answer text.
_SENTENCE_END_RE = re.compile(r'[.!?。！？](?:["”’\')\]]+)?(?:\s*\[\d+\])*(?=\s|$)')
_ABBREVIATIONS = {"mr", "mrs", "ms", "dr", "prof", "sr", "jr", "st", "vs", "etc", "e.g", "i.e", "fig", "no"}

def _is_real_sentence_end(fragment):
    # Reject list counters ("1. Value"), abbreviations and initials.
    raw = re.sub(r'(?:\s*\[\d+\])+\s*$', '', fragment).rstrip()
    raw = raw.rstrip("\"'”’)]")
    if re.fullmatch(r'\s*(?:[-*+]\s*)?\d+\.', raw):
        return False
    ending = re.search(r'([A-Za-z][A-Za-z.]*)\.$', raw)
    if ending and ending.group(1).lower() in _ABBREVIATIONS:
        return False
    if re.search(r'\b(?:[A-Za-z]\.){2,}$', raw):
        return False
    return True

def split_answer_into_claims(answer):
    """Deterministic, no extra dependencies. Original spans are stored by Python.

    Split at line boundaries (including list items) and clear sentence-ending
    punctuation. Semicolon-connected facts stay together and must ALL be
    supported for the whole unit to be supported. No LLM-generated spans.
    """
    claims = []
    for line in str(answer).splitlines():
        line = line.strip()
        if not line or re.fullmatch(r'```\w*|[-*_]{3,}', line):
            continue
        begin = 0
        for match in _SENTENCE_END_RE.finditer(line):
            piece = line[begin:match.end()].strip()
            if piece and _is_real_sentence_end(piece):
                claims.append(piece)
                begin = match.end()
        rest = line[begin:].strip()
        if rest:
            claims.append(rest)
    if not claims:
        raise ValueError("Grounding input contains no nonempty answer text to assess")
    return [{"claim_id": i, "sentence": sentence} for i, sentence in enumerate(claims, 1)]

def grounding_input(row, selected_claims=None):
    item = data[int(row["example_id"])]
    claims = selected_claims if selected_claims is not None else split_answer_into_claims(row["answer"])
    numbered = "\n".join(f"[claim_id={c['claim_id']}] {c['sentence']}" for c in claims)
    return (f"QUESTION:\n{item['question']}\n\nRETRIEVED DOCUMENTS:\n"
            f"{format_documents(item['search_results'])}\n\n"
            f"FULL CANDIDATE ANSWER (context, NOT additional claims):\n{row['answer']}\n\n"
            f"PYTHON-SEGMENTED CLAIMS TO ASSESS (exactly these IDs):\n{numbered}")

RECALL_SYSTEM = """Determine whether the candidate answer correctly includes the official reference answer (semantically, not by literal string overlap).
Do not reward a candidate that only lists a correct reference answer as an unresolved alternative among false options.
Ignore harmless paraphrases. Return strict JSON: {"recall": 0, "reason": "short explanation"}. 1=includes correctly; 0=does not.
"""

def recall_input(row):
    i = int(row["example_id"])
    return (f"QUESTION: {data[i]['question']}\nREFERENCE ANSWER: {reference_answers[i]}\n"
            f"CANDIDATE ANSWER: {row['answer']}")

print("Judge prompt preview (may contain GOLD TYPE for evaluation, never for classification):")
print(behavior_input(outputs.iloc[0])[:1400])


Judge prompt preview (may contain GOLD TYPE for evaluation, never for classification):
QUESTION:
when was coffee first made into a drink?

GOLD TYPE (evaluation only): Complementary information
DEFINITION: Different sources give compatible answers or meaningful facets of an underspecified query.
EXPECTED BEHAVIOR: Reconcile the distinct, compatible and QUESTION-RELEVANT answers or interpretations; do not invent an artificial disagreement or require unrelated trivia.

BEHAVIORAL EXAMPLES (not evidence):
Positive: Q: What does an artist membership include? One notice lists workshops; another lists studio access. Good: Mention both relevant benefits together.
Negative: Bad: Mention workshops alone despite the explicit question about total membership benefits.
Positive: Q: How can guests reach the island? Ferry and licensed shuttle are both valid routes. Good: Present both options concisely.
Negative: Bad: Treat the options as mutually exclusive or give unrelated transport trivia.

RETRIEV

## 5. Independent, resumable Judges

Three Judge caches use stable keys based on input content, model, rubric, and dataset. Grounding preserves the existing strict-schema claim-ID prompt and cache key so completed evaluations can be reused without new API calls.


In [5]:
def json_hash(obj):
    return hashlib.sha256(json.dumps(obj, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

outputs = outputs.copy()

def judgment_key(kind, row):
    core = {"kind": kind, "model": EVALUATOR_MODEL, "stage2_config_hash": row["config_hash"],
            "dataset_hash": sha, "example_id": int(row["example_id"])}
    if kind == "behavior":
        core.update(version=RUBRIC_VERSION, system=BEHAVIOR_SYSTEM, user=behavior_input(row))
    elif kind == "grounding":
        core.update(version=GROUNDING_VERSION, splitter=GROUNDING_SPLITTER_VERSION,
                     claims_per_call=GROUNDING_CLAIMS_PER_CALL,
                     schema="dynamic_required_claim_keys_with_exact_claim_id_enum",
                     system=GROUNDING_SYSTEM, user=grounding_input(row))
    elif kind == "recall":
        core.update(version=RECALL_VERSION, system=RECALL_SYSTEM, user=recall_input(row))
    else:
        raise ValueError(kind)
    return json_hash(core)

for kind in ("behavior", "grounding"):
    outputs[kind+"_key"] = [judgment_key(kind, r) for _, r in outputs.iterrows()]
outputs["recall_key"] = [judgment_key("recall", r) if int(r["example_id"]) in reference_answers else None
                          for _, r in outputs.iterrows()]

def call_json(client, system, user, tries=3, response_format=None):
    # Grounding uses strict schema; Behavior and Recall use JSON objects.
    response_format = response_format or {"type": "json_object"}
    for attempt in range(tries):
        try:
            resp = client.chat.completions.create(
                model=EVALUATOR_MODEL, temperature=0, response_format=response_format,
                messages=[{"role": "system", "content": system}, {"role": "user", "content": user}]
            )
            msg = resp.choices[0].message
            if getattr(msg, "refusal", None):
                raise ValueError(f"Judge refused to assess: {msg.refusal}")
            if not msg.content:
                raise ValueError("Judge returned empty text")
            return json.loads(msg.content)
        except Exception:
            if attempt == tries - 1:
                raise
            time.sleep(2**attempt)

def evaluate_behavior(client, row):
    obj = call_json(client, BEHAVIOR_SYSTEM, behavior_input(row))
    assert type(obj.get("adherence")) is int and obj["adherence"] in (0,1), obj
    assert isinstance(obj.get("reason"), str), obj
    return {"expected_behavior_adherence":obj["adherence"],
            "behavior_reason":obj["reason"], "missing_behavior": str(obj.get("missing_behavior") or "")}

def grounding_response_format(claim_ids):
    """Require exactly one schema-defined assessment per Python claim."""
    claim_ids = [int(i) for i in claim_ids]
    if not claim_ids or len(set(claim_ids)) != len(claim_ids):
        raise ValueError("claim_ids must be unique and nonempty")
    props = {}
    for i in claim_ids:
        props[f"claim_{i}"] = {
            "type": "object", "additionalProperties": False,
            "properties": {
                "claim_id": {"type": "integer", "enum": [i]},
                "status": {"type": "string", "enum": ["supported", "unsupported", "contradicted", "nonfactual"]},
                "evidence_doc_ids": {"type": "array", "items": {"type": "integer"}},
            },
            "required": ["claim_id", "status", "evidence_doc_ids"],
        }
    return {
        "type": "json_schema",
        "json_schema": {
            "name": "grounding_fixed_claim_ids",
            "strict": True,
            "schema": {
                "type": "object", "additionalProperties": False,
                "properties": {
                    "assessments": {
                        "type": "object", "additionalProperties": False,
                        "properties": props, "required": list(props),
                    }
                },
                "required": ["assessments"],
            },
        },
    }

def _validate_grounding_assessments(obj, expected_ids, document_count):
    """Never silently drop extra IDs or invent judgments for missing IDs."""
    if not isinstance(obj, dict) or not isinstance(obj.get("assessments"), dict):
        raise ValueError("JSON must contain a keyed assessments object")
    assessments = obj["assessments"]
    expected = {f"claim_{i}" for i in expected_ids}
    if set(assessments) != expected:
        missing = sorted(expected - set(assessments))
        extra = sorted(set(assessments) - expected)
        raise ValueError(f"Incorrect assessment claim keys: missing={missing}, extra={extra}")
    by_id = {}
    for i in expected_ids:
        entry = assessments[f"claim_{i}"]
        if not isinstance(entry, dict) or type(entry.get("claim_id")) is not int or entry["claim_id"] != i:
            raise ValueError(f"Incorrect claim_id inside claim_{i}: {entry!r}")
        status = entry.get("status")
        if status not in {"supported", "unsupported", "contradicted", "nonfactual"}:
            raise ValueError(f"Invalid status for claim_{i}: {status!r}")
        doc_ids = entry.get("evidence_doc_ids")
        if not isinstance(doc_ids, list) or any(
            type(v) is not int or v < 1 or v > document_count for v in doc_ids
        ):
            raise ValueError(f"Invalid evidence_doc_ids for claim_{i}: {doc_ids!r}")
        if len(set(doc_ids)) != len(doc_ids):
            raise ValueError(f"Duplicate evidence_doc_ids for claim_{i}: {doc_ids!r}")
        if status == "supported" and not doc_ids:
            raise ValueError(f"Supported claim_{i} has no supporting document")
        if status != "supported" and doc_ids:
            raise ValueError(f"Non-supported claim_{i} must use empty evidence_doc_ids")
        by_id[i] = entry
    return by_id

def evaluate_grounding(client, row):
    # Preserve original Python-segmented text and original claim IDs throughout.
    claims = split_answer_into_claims(row["answer"])
    docs = data[int(row["example_id"])]["search_results"]
    by_id = {}
    for offset in range(0, len(claims), GROUNDING_CLAIMS_PER_CALL):
        chunk = claims[offset : offset + GROUNDING_CLAIMS_PER_CALL]
        ids = [c["claim_id"] for c in chunk]
        fmt = grounding_response_format(ids)
        user_prompt = grounding_input(row, selected_claims=chunk)
        for attempt in range(3):
            extra = "" if attempt == 0 else (
                "\n\nThe last response failed semantic validation. "
                "Use only real supporting document IDs; supported needs at least one, "
                "and all other statuses need []. Evaluate the SAME supplied claim_ids."
            )
            obj = call_json(client, GROUNDING_SYSTEM, user_prompt + extra, response_format=fmt)
            try:
                chunk_by_id = _validate_grounding_assessments(obj, ids, len(docs))
                by_id.update(chunk_by_id)
                break
            except ValueError as exc:
                print(f"Grounding validation failed (example_id={int(row['example_id'])}, "
                      f"claim_ids={ids}, attempt {attempt + 1}/3): {exc}", flush=True)
                if attempt == 2:
                    raise ValueError(
                        f"Grounding Judge failed validation after 3 attempts "
                        f"(example_id={int(row['example_id'])}, claim_ids={ids}): {exc}"
                    ) from exc
    if set(by_id) != {c["claim_id"] for c in claims}:
        raise ValueError("Internal error: missing claim assessments after aggregation")

    processed = []
    for claim in claims:
        claim_id, sentence = claim["claim_id"], claim["sentence"]
        assessment = by_id[claim_id]
        status = assessment["status"]
        ids = sorted(assessment["evidence_doc_ids"])
        processed.append({"claim_id": claim_id, "sentence": sentence, "status": status,
                          "evidence_doc_ids": ids})
    factual = [x for x in processed if x["status"] != "nonfactual"]
    supported = sum(x["status"] == "supported" for x in factual)
    total = len(factual)
    return {"factual_sentence_count": total, "supported_sentence_count": supported,
            "factual_grounding": supported / total if total else None,
            "sentence_assessments": processed, "grounding_reason": ""}

def evaluate_recall(client,row):
    obj=call_json(client,RECALL_SYSTEM,recall_input(row))
    assert type(obj.get("recall")) is int and obj["recall"] in (0,1),obj
    return {"answer_recall":obj["recall"],"recall_reason":str(obj.get("reason") or "")}

CACHE_MAP = {"behavior":BEHAVIOR_CACHE,"grounding":GROUNDING_CACHE,"recall":RECALL_CACHE}
all_cache={}
for kind,path in CACHE_MAP.items():
    needed = outputs.loc[outputs[kind+"_key"].notna()].drop_duplicates(kind+"_key")
    valid_keys = set(needed[kind+"_key"])
    prior = read_jsonl(path) if path.is_file() else []
    got={}
    for entry in prior:
        key=entry.get("key")
        if key in got or key not in valid_keys:
            raise ValueError(f"Cache {path} contains duplicate or outdated keys; rename it before continuing")
        got[key]=entry
    all_cache[kind]=got
    print(kind, "saved",len(got),"/",len(needed))

if RUN_JUDGE_API:
    key=os.getenv("OPENAI_API_KEY") or getpass.getpass("OPENAI_API_KEY: ")
    client=OpenAI(api_key=key)
    for kind,path in CACHE_MAP.items():
        kcol=kind+"_key"
        todo=outputs.loc[outputs[kcol].notna()].drop_duplicates(kcol)
        for _,row in todo.iterrows():
            k=row[kcol]
            if k in all_cache[kind]:
                continue
            result=(evaluate_behavior(client,row) if kind=="behavior" else
                    evaluate_grounding(client,row) if kind=="grounding" else evaluate_recall(client,row))
            entry={"key":k,"example_id":int(row["example_id"]),"judge_model":EVALUATOR_MODEL,"kind":kind,**result}
            with open(path,"a",encoding="utf-8") as f:
                f.write(json.dumps(entry,ensure_ascii=False)+"\n")
                f.flush()
            all_cache[kind][k]=entry
            print(kind, "saved",len(all_cache[kind]),"/",len(todo),flush=True)
else:
    print("Judges disabled: no API calls. Cached judgments (if any) can still be summarized.")


behavior saved 0 / 165
grounding saved 0 / 165
recall saved 0 / 74
behavior saved 1 / 165
behavior saved 2 / 165
behavior saved 3 / 165
behavior saved 4 / 165
behavior saved 5 / 165
behavior saved 6 / 165
behavior saved 7 / 165
behavior saved 8 / 165
behavior saved 9 / 165
behavior saved 10 / 165
behavior saved 11 / 165
behavior saved 12 / 165
behavior saved 13 / 165
behavior saved 14 / 165
behavior saved 15 / 165
behavior saved 16 / 165
behavior saved 17 / 165
behavior saved 18 / 165
behavior saved 19 / 165
behavior saved 20 / 165
behavior saved 21 / 165
behavior saved 22 / 165
behavior saved 23 / 165
behavior saved 24 / 165
behavior saved 25 / 165
behavior saved 26 / 165
behavior saved 27 / 165
behavior saved 28 / 165
behavior saved 29 / 165
behavior saved 30 / 165
behavior saved 31 / 165
behavior saved 32 / 165
behavior saved 33 / 165
behavior saved 34 / 165
behavior saved 35 / 165
behavior saved 36 / 165
behavior saved 37 / 165
behavior saved 38 / 165
behavior saved 39 / 165
behavi

## 6. Export the three scores

Export one record per answer and report only Expected Behavior, Answer Recall, and Factual Grounding. Historical Citation scores are ignored even if stored in existing Grounding caches. Evidence-document IDs and claim-level status remain available for audit.


In [6]:
ready = all(set(outputs.loc[outputs[k + "_key"].notna(), k + "_key"]).issubset(all_cache[k])
            for k in CACHE_MAP)
print("All independent judgments complete:", ready)
if ready:
    records = []
    for _, r in outputs.iterrows():
        row = r.to_dict()
        for kind, fields in {
            "behavior": ("expected_behavior_adherence", "behavior_reason", "missing_behavior"),
            "recall": ("answer_recall", "recall_reason"),
            "grounding": ("factual_sentence_count", "supported_sentence_count", "grounding_reason"),
        }.items():
            key = r[kind + "_key"]
            item = all_cache[kind].get(key, {}) if isinstance(key, str) else {}
            row.update({name: item[name] for name in fields if name in item})
            if kind == "grounding":
                # Compatible with earlier caches; no need to rerun the Grounding API.
                row["factual_grounding"] = item.get("factual_grounding", item.get("factual_support"))
                if "sentence_assessments" in item:
                    row["sentence_assessments"] = [
                        {name: claim[name] for name in ("claim_id", "sentence", "status", "evidence_doc_ids")
                         if name in claim}
                        for claim in item["sentence_assessments"]
                    ]
        row.setdefault("answer_recall", np.nan)
        records.append(row)
    scored = pd.DataFrame(records)
    assert scored.expected_behavior_adherence.isin([0, 1]).all()
    assert scored.answer_recall.dropna().isin([0, 1]).all()
    assert scored.factual_grounding.dropna().between(0, 1).all()
    assert not any("citation_" in col for col in scored.columns), "Unexpected legacy Citation score"
    scored.to_csv(SCORES_PATH, index=False)
    print("Saved:", SCORES_PATH, "rows:", len(scored))
    display(scored[["example_id", "condition", "expected_behavior_adherence",
                    "answer_recall", "factual_grounding"]].head(12))
else:
    if "scored" in globals():
        del scored  # Do not show scores from a previous incomplete run.
    print("Scoring incomplete. Set RUN_JUDGE_API=True to evaluate missing items, then rerun.")


All independent judgments complete: True
Saved: /Users/GXM/03-trimester 2/AIML339/Assignment/7/rag-spread 新数据/rag_test_answer_scores.csv rows: 288


,example_id,condition,expected_behavior_adherence,answer_recall,factual_grounding
0,4,standard,0,NaN,1.0
1,4,llm_pipeline,1,NaN,1.0
2,4,bert,1,NaN,1.0
3,4,oracle,1,NaN,1.0
4,7,standard,1,1.0,1.0
5,7,llm_pipeline,1,1.0,1.0
6,7,bert,1,1.0,1.0
7,7,oracle,1,1.0,1.0
8,12,standard,1,0.0,1.0
9,12,llm_pipeline,1,0.0,1.0


## 7. Per-condition and per-class metrics

Report the three agreed evaluation outcomes with their own denominators. Answer Recall is defined only for questions with a verified official reference answer.


In [8]:
if "scored" in globals():
    METRICS = ["expected_behavior_adherence", "answer_recall", "factual_grounding"]

    def summarize(columns):
        summary = scored.groupby(columns, dropna=False).agg(
            n_answers=("example_id", "size"),
            **{m + "_mean": (m, "mean") for m in METRICS},
            **{m + "_n": (m, "count") for m in METRICS},
        )
        return summary.reset_index()

    summary = summarize("condition")
    per_class = summarize(["gold_label", "condition"])
    summary.to_csv(SUMMARY_PATH, index=False)
    per_class.to_csv(PER_CLASS_PATH, index=False)
    display(summary.round(4))
    display(per_class.round(4))
    print("Summaries:", SUMMARY_PATH, PER_CLASS_PATH)


,condition,n_answers,expected_behavior_adherence_mean,answer_recall_mean,factual_grounding_mean,expected_behavior_adherence_n,answer_recall_n,factual_grounding_n
0,bert,72,0.7500,0.7429,0.9444,72,35,72
1,llm_pipeline,72,0.7361,0.7714,0.9398,72,35,72
2,oracle,72,0.7917,0.7714,0.9398,72,35,72
3,standard,72,0.6111,0.7143,0.9587,72,35,72


,gold_label,condition,n_answers,expected_behavior_adherence_mean,answer_recall_mean,factual_grounding_mean,expected_behavior_adherence_n,answer_recall_n,factual_grounding_n
0,Complementary information,bert,18,0.6667,NaN,1.0000,18,0,18
1,Complementary information,llm_pipeline,18,0.6111,NaN,0.9815,18,0,18
2,Complementary information,oracle,18,0.6667,NaN,0.9815,18,0,18
3,Complementary information,standard,18,0.6111,NaN,0.9735,18,0,18
4,Conflict due to misinformation,bert,1,1.0000,1.0000,1.0000,1,1,1
5,Conflict due to misinformation,llm_pipeline,1,1.0000,1.0000,1.0000,1,1,1
6,Conflict due to misinformation,oracle,1,1.0000,1.0000,1.0000,1,1,1
7,Conflict due to misinformation,standard,1,1.0000,1.0000,1.0000,1,1,1
8,Conflict due to outdated information,bert,10,0.7000,0.5000,1.0000,10,10,10
9,Conflict due to outdated information,llm_pipeline,10,0.7000,0.5000,1.0000,10,10,10


Summaries: /Users/GXM/03-trimester 2/AIML339/Assignment/7/rag-spread 新数据/rag_test_condition_summary.csv /Users/GXM/03-trimester 2/AIML339/Assignment/7/rag-spread 新数据/rag_test_per_class_summary.csv


## 8. Paired comparison

For the same questions, compare the three outcome scores with 95% paired-bootstrap confidence intervals. Per-class sample sizes, especially Misinformation, are small.


In [9]:
if "scored" in globals():
    rng=np.random.default_rng(SEED)
    PAIRS=[("bert","standard"),("bert","llm_pipeline"),("llm_pipeline","standard"),
           ("oracle","standard"),("oracle","bert"),("oracle","llm_pipeline")]
    paired_rows=[]
    for metric in METRICS:
        wide=scored.pivot(index="example_id",columns="condition",values=metric)
        for a,b in PAIRS:
            delta=(wide[a]-wide[b]).dropna().to_numpy(dtype=float)
            if len(delta):
                bootstrap=rng.choice(delta,size=(2000,len(delta)),replace=True).mean(axis=1)
                lo,hi=np.quantile(bootstrap,[0.025,0.975])
                mean=float(delta.mean())
            else:
                lo=hi=mean=np.nan
            paired_rows.append({"metric":metric,"comparison":a+" - "+b,
                                "n_question_pairs":len(delta),"mean_difference":mean,
                                "bootstrap_ci95_low":lo,"bootstrap_ci95_high":hi,
                                "a_higher_count":int((delta>0).sum()),"equal_count":int((delta==0).sum()),
                                "a_lower_count":int((delta<0).sum())})
    paired=pd.DataFrame(paired_rows)
    paired.to_csv(PAIRED_PATH,index=False)
    display(paired.round(3))

,metric,comparison,n_question_pairs,mean_difference,bootstrap_ci95_low,bootstrap_ci95_high,a_higher_count,equal_count,a_lower_count
0,expected_behavior_adherence,bert - standard,72,0.139,0.056,0.236,11,60,1
1,expected_behavior_adherence,bert - llm_pipeline,72,0.014,-0.042,0.069,3,67,2
2,expected_behavior_adherence,llm_pipeline - standard,72,0.125,0.042,0.208,10,61,1
3,expected_behavior_adherence,oracle - standard,72,0.181,0.097,0.278,13,59,0
4,expected_behavior_adherence,oracle - bert,72,0.042,0.000,0.097,3,69,0
5,expected_behavior_adherence,oracle - llm_pipeline,72,0.056,0.014,0.111,4,68,0
6,answer_recall,bert - standard,35,0.029,0.000,0.086,1,34,0
7,answer_recall,bert - llm_pipeline,35,-0.029,-0.086,0.000,0,34,1
8,answer_recall,llm_pipeline - standard,35,0.057,0.000,0.143,2,33,0
9,answer_recall,oracle - standard,35,0.057,0.000,0.143,2,33,0


## 9. BERT and prompted LLM classification

Compare both classifiers on the same gold labels, using five-class Accuracy/Macro-F1 and the four-class Macro-F1 excluding Misinformation.


In [10]:
FOUR_LABELS = [s for s in LABELS if s != "Conflict due to misinformation"]

def classification_metrics(y_true, y_pred):
    y_true, y_pred = list(y_true), list(y_pred)
    if not set(y_true + y_pred).issubset(LABELS):
        raise ValueError("Labels must use the exact five canonical class names")
    if len(y_true) != len(y_pred) or not y_true:
        raise ValueError("Inputs must have equal nonzero lengths")
    return {
        "n": len(y_true),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1_5class": float(f1_score(y_true, y_pred, labels=LABELS,
                                           average="macro", zero_division=0)),
        "macro_f1_4class": float(f1_score(y_true, y_pred, labels=FOUR_LABELS,
                                           average="macro", zero_division=0)),
    }

if "scored" in globals():
    cls=scored[scored.condition=="standard"][["example_id","question","gold_label",
        "predicted_label","llm_predicted_label"]].sort_values("example_id").copy()
    cls["bert_correct"] = cls.gold_label == cls.predicted_label
    cls["llm_correct"] = cls.gold_label == cls.llm_predicted_label
    assert len(cls)==len(expected_ids)
    records=[]
    for method,col in [("Spread-only BERT","predicted_label"),("Prompted LLM","llm_predicted_label")]:
        truth=cls.gold_label.tolist(); pred=cls[col].tolist()
        assert set(pred)<=set(LABELS)
        records.append({"classifier": method, **classification_metrics(truth, pred)})
    comparison=pd.DataFrame(records)
    cls.to_csv(CLASSIFIER_PREDICTIONS_PATH,index=False)
    comparison.to_csv(CLASSIFIER_PATH,index=False)
    display(comparison.round(4))


,classifier,n,accuracy,macro_f1_5class,macro_f1_4class
0,Spread-only BERT,72,0.6389,0.4925,0.6157
1,Prompted LLM,72,0.5278,0.4047,0.5059
